#Import Library

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
from sklearn.metrics import silhouette_samples, silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import KMeans
sns.set()
%matplotlib inline
sns.set()
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler, OneHotEncoder, OrdinalEncoder, PolynomialFeatures
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.animation as animation
from mpl_toolkits.mplot3d import Axes3D
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.metrics.pairwise import cosine_similarity
import missingno as msno
from sklearn.metrics import silhouette_score
from sklearn.metrics import pairwise_distances
from sklearn.metrics import davies_bouldin_score
from sklearn.metrics import calinski_harabasz_score
from sklearn import preprocessing
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN

#Read Data

In [2]:
df = pd.read_excel('https://raw.github.com/nadiraaini77/python-data-analyst/master/Project4/clustering.xlsx')

In [3]:
df.head()

,Unnamed: 0,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,...,Arrival Delay in Minutes,satisfaction,total_delay,travel_disctance_category,age_group,avg_service_score,avg_digital_score,avg_comfort_score,avg_airport_experience_score,delay_category
0,0,70172,Male,Loyal Customer,13,Personal Travel,Eco Plus,460,3,4,...,18.0,neutral or dissatisfied,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,5047,Male,disloyal Customer,25,Business travel,Business,235,3,2,...,6.0,neutral or dissatisfied,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,110028,Female,Loyal Customer,26,Business travel,Business,1142,2,2,...,0.0,satisfied,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,24026,Female,Loyal Customer,25,Business travel,Business,562,2,5,...,9.0,neutral or dissatisfied,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,119299,Male,Loyal Customer,61,Business travel,Business,214,3,3,...,0.0,satisfied,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#Data Preparation

##Checking Duplicate

In [4]:
df.duplicated().sum()

np.int64(0)

##Check Missing Value

In [5]:
df.isnull().sum()

,0
Unnamed: 0,0
id,0
Gender,0
Customer Type,0
Age,0
Type of Travel,0
Class,0
Flight Distance,0
Inflight wifi service,0
Departure/Arrival time convenient,0


In [6]:
df['Arrival Delay in Minutes'] = df['Arrival Delay in Minutes'].fillna(df['Departure Delay in Minutes'])

#Feature Engineering

In [12]:
df.describe()

,Unnamed: 0,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,...,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,total_delay,avg_service_score,avg_digital_score,avg_comfort_score,avg_airport_experience_score,delay_category
count,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,...,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,0.0,0.0,0.0,0.0,0.0
mean,51951.500000,64924.210502,39.379706,1189.448375,2.729683,3.060296,2.756901,2.976883,3.202129,3.250375,...,3.640428,3.286351,14.815618,15.245072,30.060691,NaN,NaN,NaN,NaN,NaN
std,29994.645522,37463.812252,15.114964,997.147281,1.327829,1.525075,1.398929,1.277621,1.329533,1.349509,...,1.175663,1.312273,38.230901,38.808674,76.377742,NaN,NaN,NaN,NaN,NaN
min,0.000000,1.000000,7.000000,31.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN
25%,25975.750000,32533.750000,27.000000,414.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,...,3.000000,2.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN
50%,51951.500000,64856.500000,40.000000,843.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,...,4.000000,3.000000,0.000000,0.000000,2.000000,NaN,NaN,NaN,NaN,NaN
75%,77927.250000,97368.250000,51.000000,1743.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,...,5.000000,4.000000,12.000000,13.000000,24.000000,NaN,NaN,NaN,NaN,NaN
max,103903.000000,129880.000000,85.000000,4983.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,...,5.000000,5.000000,1592.000000,1584.000000,3176.000000,NaN,NaN,NaN,NaN,NaN


##1. Total Delay

In [13]:
df['total_delay']=df['Departure Delay in Minutes']+df['Arrival Delay in Minutes']

##2. Travel Distance Category

In [14]:
def jarak (dist):
  if dist < 1000:
    return 'Short'
  elif dist <= 2500:
    return 'Medium'
  else:
    return 'Long'
df['travel_disctance_category']=df['Flight Distance'].apply(jarak)

##3. Age Group

In [15]:
def umur(age):
  if age < 25:
    return 'Young'
  elif age <= 45:
    return 'Middle'
  else:
    return 'Old'
df['age_group']=df['Age'].apply(umur)

##4. Average Service Score

In [16]:
df['avg_service_score']=(df['Inflight wifi service']+df['Inflight service']+df['Leg room service']+df['On-board service']+df['Checkin service'])/5

##5. Average Digital Score

In [17]:
df['avg_digital_score']=(df['Ease of Online booking']+df['Online boarding'])/2

##6. Average Comfort Score

In [18]:
df['avg_comfort_score']=(df['Cleanliness']+df['Departure/Arrival time convenient']+df['Inflight entertainment']+df['Food and drink'])/4

##7. Average Airport Experience Score

In [19]:
df['avg_airport_experience_score']=(df['Gate location']+df['Baggage handling'])/2

##8. Delay Category

In [20]:
def terlambat(delay):
  if delay == 0:
    return 'Ontime'
  elif delay <= 60:
    return 'Low'
  elif delay <= 180:
    return 'Medium'
  else:
    return 'High'

df['delay_category'] = df['total_delay'].apply(terlambat)

In [21]:
df.describe()

,Unnamed: 0,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,...,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,total_delay,avg_service_score,avg_digital_score,avg_comfort_score,avg_airport_experience_score
count,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,...,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000
mean,51951.500000,64924.210502,39.379706,1189.448375,2.729683,3.060296,2.756901,2.976883,3.202129,3.250375,...,3.304290,3.640428,3.286351,14.815618,15.245072,30.060691,3.281564,3.003638,3.226733,3.304358
std,29994.645522,37463.812252,15.114964,997.147281,1.327829,1.525075,1.398929,1.277621,1.329533,1.349509,...,1.265396,1.175663,1.312273,38.230901,38.808674,76.377742,0.790657,1.151505,0.954325,0.870895
min,0.000000,1.000000,7.000000,31.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.600000,0.000000,0.250000,0.500000
25%,25975.750000,32533.750000,27.000000,414.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,...,3.000000,3.000000,2.000000,0.000000,0.000000,0.000000,2.800000,2.000000,2.500000,2.500000
50%,51951.500000,64856.500000,40.000000,843.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,...,3.000000,4.000000,3.000000,0.000000,0.000000,2.000000,3.400000,3.000000,3.250000,3.500000
75%,77927.250000,97368.250000,51.000000,1743.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,...,4.000000,5.000000,4.000000,12.000000,13.000000,24.000000,3.800000,4.000000,4.000000,4.000000
max,103903.000000,129880.000000,85.000000,4983.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,...,5.000000,5.000000,5.000000,1592.000000,1584.000000,3176.000000,5.000000,5.000000,5.000000,5.000000


In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103904 entries, 0 to 103903
Data columns (total 33 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Unnamed: 0                         103904 non-null  int64  
 1   id                                 103904 non-null  int64  
 2   Gender                             103904 non-null  object 
 3   Customer Type                      103904 non-null  object 
 4   Age                                103904 non-null  int64  
 5   Type of Travel                     103904 non-null  object 
 6   Class                              103904 non-null  object 
 7   Flight Distance                    103904 non-null  int64  
 8   Inflight wifi service              103904 non-null  int64  
 9   Departure/Arrival time convenient  103904 non-null  int64  
 10  Ease of Online booking             103904 non-null  int64  
 11  Gate location                      1039

#Checking Consistency

In [23]:
df['Gender'].unique()

array(['Male', 'Female'], dtype=object)

In [24]:
df['Customer Type'].unique()

array(['Loyal Customer', 'disloyal Customer'], dtype=object)

In [25]:
df['Customer Type'] = df['Customer Type'].str.strip().str.title()

In [26]:
df['Type of Travel'].unique()

array(['Personal Travel', 'Business travel'], dtype=object)

In [27]:
df['Type of Travel'] = df['Type of Travel'].str.strip().str.title()

In [28]:
df['Class'].unique()

array(['Eco Plus', 'Business', 'Eco'], dtype=object)

In [29]:
df['satisfaction'].unique()

array(['neutral or dissatisfied', 'satisfied'], dtype=object)

In [30]:
df['satisfaction'] = df['satisfaction'].str.strip().str.title()

In [31]:
df.describe()

,Unnamed: 0,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,...,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,total_delay,avg_service_score,avg_digital_score,avg_comfort_score,avg_airport_experience_score
count,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,...,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000
mean,51951.500000,64924.210502,39.379706,1189.448375,2.729683,3.060296,2.756901,2.976883,3.202129,3.250375,...,3.304290,3.640428,3.286351,14.815618,15.245072,30.060691,3.281564,3.003638,3.226733,3.304358
std,29994.645522,37463.812252,15.114964,997.147281,1.327829,1.525075,1.398929,1.277621,1.329533,1.349509,...,1.265396,1.175663,1.312273,38.230901,38.808674,76.377742,0.790657,1.151505,0.954325,0.870895
min,0.000000,1.000000,7.000000,31.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.600000,0.000000,0.250000,0.500000
25%,25975.750000,32533.750000,27.000000,414.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,...,3.000000,3.000000,2.000000,0.000000,0.000000,0.000000,2.800000,2.000000,2.500000,2.500000
50%,51951.500000,64856.500000,40.000000,843.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,...,3.000000,4.000000,3.000000,0.000000,0.000000,2.000000,3.400000,3.000000,3.250000,3.500000
75%,77927.250000,97368.250000,51.000000,1743.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,...,4.000000,5.000000,4.000000,12.000000,13.000000,24.000000,3.800000,4.000000,4.000000,4.000000
max,103903.000000,129880.000000,85.000000,4983.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,...,5.000000,5.000000,5.000000,1592.000000,1584.000000,3176.000000,5.000000,5.000000,5.000000,5.000000


In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103904 entries, 0 to 103903
Data columns (total 33 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Unnamed: 0                         103904 non-null  int64  
 1   id                                 103904 non-null  int64  
 2   Gender                             103904 non-null  object 
 3   Customer Type                      103904 non-null  object 
 4   Age                                103904 non-null  int64  
 5   Type of Travel                     103904 non-null  object 
 6   Class                              103904 non-null  object 
 7   Flight Distance                    103904 non-null  int64  
 8   Inflight wifi service              103904 non-null  int64  
 9   Departure/Arrival time convenient  103904 non-null  int64  
 10  Ease of Online booking             103904 non-null  int64  
 11  Gate location                      1039

In [33]:
df.columns=['no','id','gender','customer type','age','type of travel','class','flight distance',
            'inflight wifi service','departure/arrival time convenient','ease of online booking',
            'gate location','food and drink','online boarding','seat comfort','inflight entertainment',
            'on-board service','leg room service','baggage handling','checkin service','inflight service',
            'cleanliness','departure delay in minutes','arrival delay in minutes','satisfaction','total delay',
            'travel distance category','age group','avg service score','avg digital score','avg comfort score',
            'avg airport experience score','delay category']

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103904 entries, 0 to 103903
Data columns (total 33 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   no                                 103904 non-null  int64  
 1   id                                 103904 non-null  int64  
 2   gender                             103904 non-null  object 
 3   customer type                      103904 non-null  object 
 4   age                                103904 non-null  int64  
 5   type of travel                     103904 non-null  object 
 6   class                              103904 non-null  object 
 7   flight distance                    103904 non-null  int64  
 8   inflight wifi service              103904 non-null  int64  
 9   departure/arrival time convenient  103904 non-null  int64  
 10  ease of online booking             103904 non-null  int64  
 11  gate location                      1039

#Clustering

In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [36]:
numeric_cols = [
    'age', 'flight distance', 'inflight wifi service', 'departure/arrival time convenient',
    'ease of online booking', 'gate location', 'food and drink', 'online boarding',
    'seat comfort', 'inflight entertainment', 'on-board service', 'leg room service',
    'baggage handling', 'checkin service', 'inflight service', 'cleanliness',
    'departure delay in minutes', 'arrival delay in minutes', 'total delay', 'avg service score',
    'avg digital score', 'avg comfort score', 'avg airport experience score'
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col]).fillna(0)

##Encoding

In [37]:
fitur_dipilih = numeric_cols + ['gender', 'customer type', 'type of travel', 'class','satisfaction','travel distance category','age group','delay category']
df_model = df[fitur_dipilih].copy()

class_map = {'Eco': 0, 'Eco Plus': 1, 'Business': 2}
df_model['class'] = df_model['class'].map(class_map)

df_encoded = pd.get_dummies(df_model, columns=['gender', 'customer type', 'type of travel', 'satisfaction', 'travel distance category', 'age group', 'delay category'], drop_first=True, dtype=int)

print("\n1. Proses Encoding Selesai. Dimensi Data Saat Ini:", df_encoded.shape)


1. Proses Encoding Selesai. Dimensi Data Saat Ini: (103904, 35)


##Cek Multikolinearitas

In [38]:
vif_data = pd.DataFrame()
vif_data["Fitur"] = df_encoded.columns
vif_data["VIF"] = [variance_inflation_factor(df_encoded.values, i) for i in range(df_encoded.shape[1])]

print("\n2. Hasil Uji VIF (Sebelum PCA):")
print(vif_data.sort_values(by="VIF", ascending=False).to_string(index=False))


2. Hasil Uji VIF (Sebelum PCA):
                            Fitur       VIF
            inflight wifi service       inf
           ease of online booking       inf
departure/arrival time convenient       inf
                avg comfort score       inf
                avg digital score       inf
                    gate location       inf
                   food and drink       inf
                  online boarding       inf
           inflight entertainment       inf
                 on-board service       inf
                 leg room service       inf
                      cleanliness       inf
                 baggage handling       inf
                  checkin service       inf
                 inflight service       inf
         arrival delay in minutes       inf
       departure delay in minutes       inf
                      total delay       inf
                avg service score       inf
     avg airport experience score       inf
                              age 39.262270

In [39]:
#import library
import numpy as np
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [40]:
#copy dataset
df_vif_clean = df_encoded.copy()

print("Jumlah fitur awal:", df_vif_clean.shape[1])

Jumlah fitur awal: 35


In [41]:
#hitung VIF awal
vif_awal = pd.DataFrame()
vif_awal["Fitur"] = df_vif_clean.columns
vif_awal["VIF"] = [
    variance_inflation_factor(df_vif_clean.values, i)
    for i in range(df_vif_clean.shape[1])
]

vif_awal.sort_values("VIF", ascending=False)

,Fitur,VIF
2,inflight wifi service,inf
4,ease of online booking,inf
3,departure/arrival time convenient,inf
21,avg comfort score,inf
20,avg digital score,inf
5,gate location,inf
6,food and drink,inf
7,online boarding,inf
9,inflight entertainment,inf
10,on-board service,inf


In [42]:
#eliminasi VIF
print("--- Memulai Eliminasi VIF ---")

while True:

    vif_list = []

    for i in range(df_vif_clean.shape[1]):
        try:
            vif_val = variance_inflation_factor(
                df_vif_clean.values, i
            )
        except:
            vif_val = 999999

        vif_list.append(vif_val)

    temp_vif = pd.DataFrame({
        "Fitur": df_vif_clean.columns,
        "VIF": vif_list
    })

    temp_vif["VIF"] = temp_vif["VIF"].replace(
        [np.inf, -np.inf],
        999999
    )

    max_row = temp_vif.loc[
        temp_vif["VIF"].idxmax()
    ]

    if max_row["VIF"] >= 10:

        print(
            f"Menghapus {max_row['Fitur']} "
            f"(VIF={max_row['VIF']:.2f})"
        )

        df_vif_clean.drop(
            columns=[max_row["Fitur"]],
            inplace=True
        )

    else:
        break

print("Eliminasi selesai")

--- Memulai Eliminasi VIF ---
Menghapus inflight wifi service (VIF=999999.00)
Menghapus departure/arrival time convenient (VIF=999999.00)
Menghapus ease of online booking (VIF=999999.00)
Menghapus gate location (VIF=999999.00)
Menghapus departure delay in minutes (VIF=999999.00)
Menghapus avg service score (VIF=406.90)
Menghapus avg comfort score (VIF=129.45)
Menghapus total delay (VIF=70.83)
Menghapus age (VIF=39.16)
Menghapus avg airport experience score (VIF=38.10)
Menghapus online boarding (VIF=29.02)
Menghapus inflight entertainment (VIF=27.73)
Menghapus delay category_Ontime (VIF=25.06)
Menghapus inflight service (VIF=20.37)
Menghapus cleanliness (VIF=17.25)
Menghapus baggage handling (VIF=15.44)
Menghapus seat comfort (VIF=13.40)
Menghapus travel distance category_Short (VIF=10.74)
Eliminasi selesai


In [43]:
df_vif_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103904 entries, 0 to 103903
Data columns (total 17 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   flight distance                  103904 non-null  int64  
 1   food and drink                   103904 non-null  int64  
 2   on-board service                 103904 non-null  int64  
 3   leg room service                 103904 non-null  int64  
 4   checkin service                  103904 non-null  int64  
 5   arrival delay in minutes         103904 non-null  float64
 6   avg digital score                103904 non-null  float64
 7   class                            103904 non-null  int64  
 8   gender_Male                      103904 non-null  int64  
 9   customer type_Loyal Customer     103904 non-null  int64  
 10  type of travel_Personal Travel   103904 non-null  int64  
 11  satisfaction_Satisfied           103904 non-null  int64  
 12  tr

In [44]:
df_vif_clean

,flight distance,food and drink,on-board service,leg room service,checkin service,arrival delay in minutes,avg digital score,class,gender_Male,customer type_Loyal Customer,type of travel_Personal Travel,satisfaction_Satisfied,travel distance category_Medium,age group_Old,age group_Young,delay category_Low,delay category_Medium
0,460,5,4,3,4,18.0,3.0,1,1,1,1,0,0,0,1,1,0
1,235,1,1,5,1,6.0,3.0,2,1,0,0,0,0,0,0,1,0
2,1142,5,4,3,4,0.0,3.5,2,0,1,0,1,1,0,0,0,0
3,562,2,2,5,1,9.0,3.5,2,0,1,0,0,0,0,0,1,0
4,214,4,3,4,3,0.0,4.0,2,1,1,0,1,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103899,192,2,3,1,2,0.0,2.0,0,0,0,0,0,0,0,1,1,0
103900,2347,2,5,5,5,0.0,4.0,2,1,1,0,1,1,1,0,0,0
103901,1995,4,3,2,5,14.0,1.0,2,1,0,0,0,1,0,0,1,0
103902,1000,1,4,5,5,0.0,1.0,0,0,0,0,0,1,0,1,0,0


#Export

In [45]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [46]:
df_vif_clean.to_excel(
    '/content/drive/MyDrive/df_vif_clean.xlsx',
    index=False
)

In [47]:
#cek file
import os

os.path.exists('/content/drive/MyDrive/df_vif_clean.xlsx')

True

#STOP

lanjut ke notebook baru (lemot soalnya)